In [1]:
!git clone https://github.com/ducbao210/Video_deepfake_detection.git

Cloning into 'Video_deepfake_detection'...
remote: Enumerating objects: 606, done.
remote: Counting objects: 100% (606/606), done.
remote: Compressing objects: 100% (334/334), done.
remote: Total 606 (delta 358), reused 496 (delta 251), pack-reused 0 (from 0)
Receiving objects: 100% (606/606), 3.84 MiB | 28.91 MiB/s, done.
Resolving deltas: 100% (358/358), done.


In [2]:
%cd Video_deepfake_detection

/kaggle/working/Video_deepfake_detection


In [3]:
!pip install -q -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 6.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 908.2/908.2 MB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 89.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 66.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.0/172.0 kB 11.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 69.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 37.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [4]:
!pip install "hydra-core>=1.3.2" "omegaconf>=2.3.0" "ml_dtypes>=0.5.0"

In [5]:
from kaggle_secrets import UserSecretsClient

try:
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")

    with open('.env', 'w') as f:
        f.write(f"HF_TOKEN={hf_token}\n")
    print("Successfully created .env file.")

except Exception as e:
    print(f"Could not find HF_TOKEN in Secrets.\n{e}")

Successfully created .env file.


In [6]:
!mkdir -p data/processed
!gdown "https://drive.google.com/uc?id=1kqGpffyfES8MURpVTQcjjmwun92n7ABA" -O data/processed.zip
!unzip -q data/processed.zip -d data/
!rm data/processed.zip

Downloading...
From (original): https://drive.google.com/uc?id=1kqGpffyfES8MURpVTQcjjmwun92n7ABA
From (redirected): https://drive.google.com/uc?id=1kqGpffyfES8MURpVTQcjjmwun92n7ABA&confirm=t&uuid=0db20819-b52e-4773-acc2-8b10e54ab245
To: /kaggle/working/Video_deepfake_detection/data/processed.zip
100%|██████████████████████████████████████| 1.12G/1.12G [00:20<00:00, 54.0MB/s]


In [7]:
!python scripts/split_dataset.py

2026-08-10 01:43:47,879 | INFO | DEEPFAKEDETECTION | Found 3431 videos with extracted frames.
2026-08-10 01:43:47,881 | INFO | DEEPFAKEDETECTION | Found 28 unique actors: ['01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28']
2026-08-10 01:43:47,881 | INFO | DEEPFAKEDETECTION | Train actors (16): ['01', '02', '03', '04', '05', '08', '09', '14', '16', '18', '20', '21', '23', '24', '26', '27']
2026-08-10 01:43:47,881 | INFO | DEEPFAKEDETECTION | Validation actors (6): ['06', '10', '12', '13', '15', '25']
2026-08-10 01:43:47,881 | INFO | DEEPFAKEDETECTION | Test actors (6): ['07', '11', '17', '19', '22', '28']
2026-08-10 01:43:47,917 | INFO | DEEPFAKEDETECTION | ========== SUMMARY ==========
2026-08-10 01:43:47,917 | INFO | DEEPFAKEDETECTION | STRATEGY: ACTOR_DISJOINT
2026-08-10 01:43:47,918 | INFO | DEEPFAKEDETECTION | TRAIN    | Total: 1215  | Real: 211  | Fake: 1004
2026-08

In [8]:
# Baseline - ConvNeXt
!python scripts/train.py model=convnext

2026-08-10 01:44:14,209 | INFO | CONVNEXT | Starting training for experiment: convnext
[*] Global random seed set to 42 for Python, NumPy, PyTorch, and cuDNN.
model.safetensors: 100%|█████████████████████| 201M/201M [00:03<00:00, 58.1MB/s]
2026-08-10 01:44:23,707 | INFO | CONVNEXT | MODEL: CONVNEXT
2026-08-10 01:44:23,707 | INFO | CONVNEXT | TOTAL PARAMETERS: 49,456,226
2026-08-10 01:44:23,708 | INFO | CONVNEXT | Training set class distribution: [211, 1004]
2026-08-10 01:44:23,708 | INFO | CONVNEXT | Class weights: [2.8791468143463135, 0.605079710483551]
2026-08-10 01:44:24,327 | INFO | CONVNEXT | No valid checkpoint found. Starting training from scratch.
2026-08-10 01:44:24,328 | INFO | CONVNEXT | Starting training for convnext on cuda...
2026-08-10 01:44:24,328 | INFO | CONVNEXT | 
[==================== Epoch 1/10 ====================]
Evaluating: 100%|██████████████████| 69/69 [00:30<00:00,  2.24it/s, loss=0.6503]
2026-08-10 01:46:13,018 | INFO | CONVNEXT | Train - ACCURACY: 0.6519 

In [9]:
# Baseline - ConvNeXt
!python scripts/train.py model=convnext

# Hybrid ConvNeXt-BiLSTM
!python scripts/train.py model=hybrid_bilstm

# Video Swin
!python scripts/train.py model=video_swin

# Knowledge Distillation - Teacher model: Video Swin - Student model: ConvNeXt
!python scripts/train_kd.py model=convnext_kd training=kd_training

# Timesformer
!python scripts/train.py model=timesformer

2026-08-10 02:04:25,639 | INFO | CONVNEXT | Starting training for experiment: convnext
[*] Global random seed set to 42 for Python, NumPy, PyTorch, and cuDNN.
2026-08-10 02:04:30,400 | INFO | CONVNEXT | MODEL: CONVNEXT
2026-08-10 02:04:30,400 | INFO | CONVNEXT | TOTAL PARAMETERS: 49,456,226
2026-08-10 02:04:30,401 | INFO | CONVNEXT | Training set class distribution: [211, 1004]
2026-08-10 02:04:30,402 | INFO | CONVNEXT | Class weights: [2.8791468143463135, 0.605079710483551]
2026-08-10 02:04:30,995 | INFO | CONVNEXT | No valid checkpoint found. Starting training from scratch.
2026-08-10 02:04:30,995 | INFO | CONVNEXT | Starting training for convnext on cuda...
2026-08-10 02:04:30,995 | INFO | CONVNEXT | 
[==================== Epoch 1/10 ====================]
Evaluating: 100%|██████████████████| 69/69 [00:32<00:00,  2.13it/s, loss=0.6503]
2026-08-10 02:06:22,667 | INFO | CONVNEXT | Train - ACCURACY: 0.6519 - BALANCED_ACCURACY: 0.4899 - PRECISION: 0.8224 - RECALL: 0.7380 - F1: 0.7780 - A

In [10]:
# Evaluate ConvNeXt
!python scripts/evaluate.py model=convnext inference.checkpoint=outputs/convnext/checkpoints/best.pth

# Evaluate Hybrid BiLSTM
!python scripts/evaluate.py model=hybrid_bilstm inference.checkpoint=outputs/hybrid_bilstm/checkpoints/best.pth

# Evaluate Video Swin
!python scripts/evaluate.py model=video_swin inference.checkpoint=outputs/video_swin/checkpoints/best.pth

# Evaluate model Student - backbone is still convnext 
!python scripts/evaluate.py model=convnext inference.checkpoint=outputs/convnext_kd/checkpoints/best.pth


# Evaluate Timeformer
!python scripts/evaluate.py model=timesformer inference.checkpoint=outputs/timesformer/checkpoints/best.pth

[*] Global random seed set to 42 for Python, NumPy, PyTorch, and cuDNN.
2026-08-10 04:25:02,365 | INFO | CONVNEXT | MODEL: CONVNEXT
2026-08-10 04:25:02,365 | INFO | CONVNEXT | Loading checkpoint from: outputs/convnext/checkpoints/best.pth
2026-08-10 04:25:02,366 | INFO | CONVNEXT | Checkpoint not found locally at outputs/convnext/checkpoints/best.pth. Attempting to download from Hugging Face...
2026-08-10 04:25:02,367 | INFO | CONVNEXT | Downloading checkpoints/convnext/best.pth from repo ducbao210/video-deepfake-detection...
checkpoints/convnext/best.pth: 100%|█████████| 322M/322M [00:07<00:00, 44.5MB/s]
2026-08-10 04:25:10,055 | INFO | CONVNEXT | Download complete! Checkpoint saved at: outputs/convnext/checkpoints/best.pth
Predicting: 100%|███████████████████████████████| 69/69 [00:33<00:00,  2.06it/s]
2026-08-10 04:25:43,992 | INFO | CONVNEXT | Optimal F1 threshold on the validation set: 0.25
Predicting: 100%|███████████████████████████████| 24/24 [00:12<00:00,  1.91it/s]
2026-08-10